In [1]:
import pandas as pd
import numpy as np
import os

# =========================
# CONFIG
# =========================
SALES_FILE = "/Users/abhigyanseth/Projects/m5_forecasting/data/sales_train_validation.csv"
CALENDAR_FILE = "/Users/abhigyanseth/Projects/m5_forecasting/data/calendar.csv"
PRICES_FILE = "/Users/abhigyanseth/Projects/m5_forecasting/data/sell_prices.csv"

OUTPUT_FOLDER = "/Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/"
STEP_SIZE = 28   # process 28 days at a time

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# =========================
# LOAD STATIC DATA
# =========================
print("Loading static datasets...")

calendar = pd.read_csv(CALENDAR_FILE)
prices = pd.read_csv(PRICES_FILE)

calendar['date'] = pd.to_datetime(calendar['date'])

# optimize
calendar = calendar[['d', 'date', 'wm_yr_wk',
                     'event_name_1', 'event_type_1',
                     'snap_CA', 'snap_TX', 'snap_WI']]

print("Loaded calendar & prices.")


# =========================
# LOAD SALES (ONLY ONCE)
# =========================
sales = pd.read_csv(SALES_FILE)

id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
d_cols = [col for col in sales.columns if col.startswith('d_')]

num_steps = len(d_cols) // STEP_SIZE + 1


# =========================
# PROCESS IN STEPS
# =========================
for i in range(num_steps):

    print(f"\nProcessing step {i+1}/{num_steps}")

    # select subset of days
    cols = d_cols[i*STEP_SIZE : (i+1)*STEP_SIZE]

    if not cols:
        continue

    temp = sales[id_vars + cols]

    # melt only this subset
    temp = temp.melt(
        id_vars=id_vars,
        value_vars=cols,
        var_name='d',
        value_name='sales'
    )

    # -------------------------
    # MERGE CALENDAR
    # -------------------------
    temp = temp.merge(calendar, on='d', how='left')

    # -------------------------
    # MERGE PRICES
    # -------------------------
    temp['store_id'] = temp['store_id'].astype(str)
    temp['item_id'] = temp['item_id'].astype(str)

    prices['store_id'] = prices['store_id'].astype(str)
    prices['item_id'] = prices['item_id'].astype(str)

    temp = temp.merge(
        prices,
        on=['store_id', 'item_id', 'wm_yr_wk'],
        how='left'
    )

    # -------------------------
    # CLEAN
    # -------------------------
    temp['date'] = pd.to_datetime(temp['date'])

    # fill missing prices
    temp['sell_price'] = temp.groupby('id')['sell_price'].ffill().bfill()

    # -------------------------
    # SAVE CHUNK
    # -------------------------
    output_path = os.path.join(OUTPUT_FOLDER, f"part_{i}.parquet")
    temp.to_parquet(output_path, index=False)

    print(f"Saved: {output_path}")

print("\n✅ Dataset built successfully (memory safe)!")

Loading static datasets...
Loaded calendar & prices.

Processing step 1/69
Saved: /Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/part_0.parquet

Processing step 2/69
Saved: /Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/part_1.parquet

Processing step 3/69
Saved: /Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/part_2.parquet

Processing step 4/69
Saved: /Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/part_3.parquet

Processing step 5/69
Saved: /Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/part_4.parquet

Processing step 6/69
Saved: /Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/part_5.parquet

Processing step 7/69
Saved: /Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/part_6.parquet

Processing step 8/69
Saved: /Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/part_7.parquet

Processing step 9/69
Saved: /Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/part_8.p

In [5]:
df = pd.read_parquet("/Users/abhigyanseth/Projects/m5_forecasting/data/final_chunks/part_0.parquet")

print(df.shape)
print(df.columns)
print(df.head())

(853720, 16)
Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'sales', 'date', 'wm_yr_wk', 'event_name_1', 'event_type_1', 'snap_CA',
       'snap_TX', 'snap_WI', 'sell_price'],
      dtype='object')
                              id        item_id    dept_id   cat_id store_id  \
0  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
1  HOBBIES_1_002_CA_1_validation  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1   
2  HOBBIES_1_003_CA_1_validation  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1   
3  HOBBIES_1_004_CA_1_validation  HOBBIES_1_004  HOBBIES_1  HOBBIES     CA_1   
4  HOBBIES_1_005_CA_1_validation  HOBBIES_1_005  HOBBIES_1  HOBBIES     CA_1   

  state_id    d  sales       date  wm_yr_wk event_name_1 event_type_1  \
0       CA  d_1      0 2011-01-29     11101         None         None   
1       CA  d_1      0 2011-01-29     11101         None         None   
2       CA  d_1      0 2011-01-29     11101         None         Non

In [7]:
dup = df.duplicated(['id', 'date']).sum()
print("Duplicate rows:", dup)

Duplicate rows: 0


In [9]:
df[['date', 'wm_yr_wk']].drop_duplicates().head(20)

,date,wm_yr_wk
0,2011-01-29,11101
30490,2011-01-30,11101
60980,2011-01-31,11101
91470,2011-02-01,11101
121960,2011-02-02,11101
152450,2011-02-03,11101
182940,2011-02-04,11101
213430,2011-02-05,11102
243920,2011-02-06,11102
274410,2011-02-07,11102


In [11]:
print("Missing price ratio:", df['sell_price'].isna().mean())

Missing price ratio: 2.3426884692873543e-06
